# Spark API Reference - Fabric Runtime 2.0

Fabric-native edition. The `spark` session and `SparkContext` are managed by Microsoft Fabric; do not create or stop them manually. Before running, select **Fabric Runtime 2.0** and attach a Lakehouse that you can write to. Any workspace and Lakehouse are supported; this checked-in notebook contains no tenant-specific IDs. The first code cell verifies Apache Spark 4.1 and prints the active Fabric context.


# Spark API reference — runnable PySpark examples

This notebook turns the official [PySpark 4.1 API Reference](https://spark.apache.org/docs/4.1.0/api/python/reference/index.html)
into an executable field guide. The upstream index contains thousands of symbols
across SQL, streaming, pandas-on-Spark, MLlib, RDDs, resources, profiling,
testing, and errors. Here, each major family receives:

- a runnable representative example when it is Microsoft Fabric Spark compatible;
- an availability note when it requires a local JVM driver or another
  separate runtime;
- links to official Spark and Fabric documentation for deeper study.

All actions are bounded. The notebook uses a remote Microsoft Fabric Spark session,
so it never relies on `SparkContext`, JVM internals, or unbounded `collect`.


## 1. API map and execution model

The most portable PySpark surface is `pyspark.sql`: `SparkSession`,
`DataFrame`, `Column`, functions, types, windows, readers, and writers.
Structured Streaming, pandas API on Spark, MLlib, RDDs, and resource APIs
have additional runtime requirements. Fabric injects a classic PySpark session with SparkContext access; use runtime context and capability checks for portable code.


In [ ]:
# Microsoft Fabric injects the Spark session.
import importlib.util
import os
import sys

import pyspark
from notebookutils import runtime
from pyspark.sql import Row, Window, functions as F, types as T

spark = globals()["spark"]
fabric_context = runtime.context
actual_environment_id = fabric_context.get("environmentId")
assert spark.version.startswith("4.1"), (
    f"Fabric Runtime 2.0 requires Apache Spark 4.1; got {spark.version}"
)
print({
    "fabric_workspace_id": fabric_context.get("currentWorkspaceId"),
    "fabric_notebook": fabric_context.get("currentNotebookName"),
    "environment_id": actual_environment_id,
    "spark": spark.version,
    "application_id": spark.sparkContext.applicationId,
})

import os
import shutil
import uuid
from pathlib import Path

print({
    "pyspark": pyspark.__version__,
    "fabric_runtime": "2.0",
    "is_remote": False,
})


## 2. `SparkSession`: lifecycle, SQL, tables, range, and version

`SparkSession` is the entry point. Fabric injects `spark`; `sql`, `table`, and `range` construct lazy DataFrames. Fabric owns the session lifecycle, so notebooks must not call `stop()`.


In [ ]:
print("server version:", spark.version)
spark.range(1, 6).select(
    "id", (F.col("id") * F.col("id")).alias("squared")
).show()


## 3. Creating DataFrames with explicit schemas

`createDataFrame` accepts Python rows, tuples, dictionaries, pandas objects,
Arrow tables, and RDDs in classic mode. Explicit `StructType` schemas make
nullability, widths, decimals, arrays, maps, and structs contractual.


In [ ]:
customer_schema = T.StructType([
    T.StructField("customer_id", T.LongType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
])
customers = spark.createDataFrame([
    (10, "Ada", "west"),
    (20, "Grace", "east"),
    (30, "Linus", "north"),
    (40, "Edsger", None),
], schema=customer_schema)
customers.printSchema()
customers.show()


## 4. Deterministic fixtures and temporary views

SQL is concise for typed fixtures; DataFrame APIs remain the main reference
surface below. Temporary views are session-scoped and let `spark.table()`
bridge SQL-created data back into fluent APIs.


In [ ]:
spark.sql("""CREATE OR REPLACE TEMP VIEW api_orders AS
SELECT *
  FROM VALUES
  (1, 10, TIMESTAMP '2025-01-03 09:15:00', 'paid',
   CAST(25.50 AS DECIMAL(12, 2)), ARRAY('gift', 'priority'),
   MAP('device', 'mobile', 'campaign', 'winter')),
  (2, 20, TIMESTAMP '2025-01-04 11:30:00', 'paid',
   CAST(40.00 AS DECIMAL(12, 2)), ARRAY('gift'),
   MAP('device', 'web', 'campaign', 'winter')),
  (3, 10, TIMESTAMP '2025-01-04 16:45:00', 'refund',
   CAST(-15.00 AS DECIMAL(12, 2)), ARRAY('priority'),
   MAP('device', 'mobile', 'campaign', 'service')),
  (4, 99, TIMESTAMP '2025-01-05 08:00:00', 'paid',
   CAST(12.00 AS DECIMAL(12, 2)), ARRAY('new'), MAP('device', 'store')),
  (5, 30, TIMESTAMP '2025-02-08 13:00:00', 'pending',
   CAST(NULL AS DECIMAL(12, 2)), ARRAY(), MAP('device', 'web'))
AS o(order_id, customer_id, event_ts, status, amount, tags, attrs)""")
orders = spark.table("api_orders")
assert orders.count() == 5


## 5. DataFrame inspection

`columns`, `dtypes`, `schema`, `printSchema`, `isEmpty`, `count`, `head`, and
`first` help inspect a relation. Actions execute the plan; metadata access
should not be confused with a free data scan.


In [ ]:
print("columns:", orders.columns)
print("dtypes:", orders.dtypes)
print("is empty:", orders.isEmpty())
print("first row:", orders.orderBy("order_id").first().asDict())


## 6. `Column` expressions, projection, aliases, and casts

Columns are immutable expression trees. Python operators create arithmetic
or predicates; `alias`, `cast`, and function calls build named projections.
Evaluation waits for an action.


In [ ]:
projected = orders.select(
    "order_id",
    F.upper("status").alias("status_upper"),
    F.coalesce("amount", F.lit(0)).alias("reporting_amount"),
    (F.col("amount") * F.lit(2)).cast("decimal(14,2)").alias("doubled"),
)
projected.orderBy("order_id").show()


## 7. Filtering and boolean composition

`filter` and `where` are synonyms. Combine predicates with `&`, `|`, and
`~`; parenthesize each comparison because Python operator precedence differs
from SQL. Use `isNull` and `isNotNull` for null checks.


In [ ]:
selected = orders.filter(
    (F.col("status") == "paid")
    & F.col("amount").isNotNull()
    & (F.col("amount") >= F.lit(20))
)
selected.select("order_id", "amount").orderBy("order_id").show()


## 8. Null handling with `DataFrame.na`

`dropna`, `fillna`, and `replace` are DataFrame conveniences. Column-level
`coalesce`, `nanvl`, `when`, and null-safe equality address expression-level
semantics. Fill values must be compatible with target column types.


In [ ]:
filled = orders.fillna({"status": "unknown"}).select(
    "order_id", "status", F.coalesce("amount", F.lit(0)).alias("amount")
)
filled.orderBy("order_id").show()


## 9. Conditional expressions

Chain `when(...).when(...).otherwise(...)` for searched conditions. Prefer
explicit output types when branches mix nulls, numerics, or decimals.


In [ ]:
classified = orders.withColumn(
    "amount_class",
    F.when(F.col("amount").isNull(), "unknown")
     .when(F.col("amount") < 0, "reversal")
     .when(F.col("amount") >= 25, "high")
     .otherwise("standard"),
)
classified.select("order_id", "amount", "amount_class").show()


## 10. Sorting, null placement, limits, and top-K

Results have no stable order without `orderBy`/`sort`. Specify direction,
null placement, and deterministic tie-breakers. Apply `limit` before
collecting top-K outputs to the driver.


In [ ]:
top = orders.orderBy(
    F.col("amount").desc_nulls_last(), F.col("order_id").asc()
).limit(3)
top.select("order_id", "amount").show()


## 11. Distinct rows and deterministic deduplication

`distinct` and `dropDuplicates` remove duplicates by row or key but do not
express a survivor policy. Use `row_number` over a deterministic window
when the latest or highest-priority record must win.


In [ ]:
versions = spark.createDataFrame([
    (1, "2025-01-01 09:00:00", "old", 1),
    (1, "2025-01-02 09:00:00", "new", 2),
    (2, "2025-01-01 10:00:00", "only", 1),
], ["id", "updated_at", "value", "priority"])
survivor_window = Window.partitionBy("id").orderBy(
    F.col("updated_at").desc(), F.col("priority").desc()
)
latest = versions.withColumn(
    "survivor_rank", F.row_number().over(survivor_window)
).filter("survivor_rank = 1").drop("survivor_rank", "priority")
latest.orderBy("id").show()


## 12. Joins: inner, outer, semi, and anti

Named-key joins merge one key column. Expression joins preserve both inputs'
keys and may require aliases. Semi/anti joins implement existence logic
without multiplying left-side rows.


In [ ]:
joined = orders.join(customers, "customer_id", "left").select(
    "order_id", "customer_id", "name", "region", "amount"
)
joined.orderBy("order_id").show()
customers.join(orders, "customer_id", "left_anti").show()


## 13. Cross joins and cardinality

`crossJoin` creates the Cartesian product. It is useful for small parameter
grids and dangerous for large relations. Estimate row growth and project
only required columns before crossing.


In [ ]:
grid = spark.range(1, 4).crossJoin(
    spark.createDataFrame([("A",), ("B",)], ["scenario"])
)
assert grid.count() == 6
grid.orderBy("id", "scenario").show()


## 14. `GroupedData` and aggregate functions

`groupBy().agg(...)` computes several metrics in one aggregation. Name each
output explicitly. `count(*)` counts rows; aggregate functions generally
ignore null inputs.


In [ ]:
metrics = orders.groupBy("status").agg(
    F.count(F.lit(1)).alias("rows"),
    F.count("amount").alias("known_amounts"),
    F.sum("amount").alias("net_amount"),
    F.round(F.avg("amount"), 2).alias("average_amount"),
    F.min("amount").alias("minimum_amount"),
    F.max("amount").alias("maximum_amount"),
)
metrics.orderBy("status").show()


## 15. Conditional aggregation and reshape patterns

Conditional aggregates create stable pivot-style schemas. `GroupedData.pivot`
is convenient when pivot values are known, but explicit conditional metrics
make names, defaults, and schema evolution easier to control.


In [ ]:
by_customer = orders.groupBy("customer_id").agg(
    F.sum(F.when(F.col("status") == "paid", F.col("amount"))
          .otherwise(F.lit(0))).alias("paid_amount"),
    F.sum(F.when(F.col("status") == "refund", F.col("amount"))
          .otherwise(F.lit(0))).alias("refund_amount"),
)
by_customer.orderBy(F.col("customer_id").asc_nulls_last()).show()


## 16. Window specifications and analytics

`Window.partitionBy`, `orderBy`, and `rowsBetween` construct reusable specs.
Window functions preserve input rows while adding aggregate, ranking, or
navigation values.


In [ ]:
running_window = Window.partitionBy("customer_id").orderBy(
    "event_ts", "order_id"
).rowsBetween(Window.unboundedPreceding, Window.currentRow)
ordered_window = Window.partitionBy("customer_id").orderBy(
    "event_ts", "order_id"
)
windowed = orders.select(
    "customer_id", "order_id", "amount",
    F.sum("amount").over(running_window).alias("running_amount"),
    F.lag("amount").over(ordered_window).alias("previous_amount"),
    F.row_number().over(ordered_window).alias("sequence"),
)
windowed.orderBy(F.col("customer_id").asc_nulls_last(), "order_id").show()


## 17. Arrays, maps, and structs

Collection functions avoid exploding data when row cardinality should stay
fixed. Use `element_at`, `try_element_at`, `array_contains`, `map_keys`, and
`struct` to preserve and inspect nested values.


In [ ]:
nested = orders.select(
    "order_id",
    F.expr("size(tags)").alias("tag_count"),
    F.expr("try_element_at(tags, 1)").alias("first_tag"),
    F.expr("array_contains(tags, 'priority')").alias("priority"),
    F.expr("element_at(attrs, 'device')").alias("device"),
    F.expr("named_struct('status', status, 'amount', amount)")
     .alias("summary"),
)
nested.orderBy("order_id").show(truncate=False)


## 18. Higher-order functions

`transform`, `filter`, `exists`, `forall`, and `aggregate` operate inside
arrays with lambdas. `expr` is the stable escape hatch for `transform`,
`filter`, and `aggregate`; this endpoint requires full SQL syntax when the
`exists` function would otherwise be parsed as the subquery keyword.


In [ ]:
spark.range(1).select(
    F.expr("ARRAY(1,2,3,4)").alias("values"),
    F.expr("transform(ARRAY(1,2,3), x -> x * 10)").alias("scaled"),
    F.expr("filter(ARRAY(1,2,3,4), x -> x % 2 = 0)").alias("evens"),
    F.expr("aggregate(ARRAY(1,2,3), 0, (acc, x) -> acc + x)")
     .alias("total"),
).show()


## 19. String and regular-expression functions

Functions include case conversion, trimming, padding, splitting,
replacement, substring extraction, and regular expressions. Escape regexes
for Python before Spark receives them.


In [ ]:
orders.select(
    "order_id",
    F.expr("upper(status)").alias("upper_status"),
    F.expr("concat_ws('-', status, CAST(order_id AS STRING))")
     .alias("label"),
    F.expr("regexp_extract('order-2025-0042', '(\\d{4})-(\\d+)', 2)")
     .alias("suffix"),
).orderBy("order_id").show()


## 20. Dates, timestamps, and intervals

Prefer typed temporal columns. Functions parse, truncate, format, extract,
and shift calendar values. SQL expressions remain useful for interval
literals and syntax not represented by a dedicated Python wrapper.


In [ ]:
calendar = orders.select(
    "order_id", "event_ts",
    F.to_date("event_ts").alias("event_date"),
    F.date_trunc("month", "event_ts").alias("month_start"),
    F.dayofweek("event_ts").alias("day_of_week"),
    F.expr("event_ts + INTERVAL 2 HOURS").alias("service_deadline"),
)
calendar.orderBy("order_id").show()


## 21. JSON and typed complex data

`from_json` requires a schema; `to_json` serializes structs, arrays, or maps.
Typed parsing catches malformed contracts earlier than ad-hoc string paths.


In [ ]:
payloads = spark.createDataFrame([
    ('{"id":7,"tags":["a","b"]}',),
], ["raw"])
payload_schema = T.StructType([
    T.StructField("id", T.LongType()),
    T.StructField("tags", T.ArrayType(T.StringType())),
])
print("payload schema:", payload_schema.simpleString())
parsed = payloads.withColumn(
    "value",
    F.expr("from_json(raw, 'STRUCT<id: BIGINT, tags: ARRAY<STRING>>')"),
)
parsed.select(
    "value",
    F.expr("to_json(value)").alias("round_trip"),
).show(truncate=False)


## 22. Set operations

`union`/`unionAll` preserve duplicates. `intersect`, `intersectAll`,
`subtract`, and `exceptAll` implement set or multiset operations. Inputs
must have compatible columns in compatible order unless using by-name union.


In [ ]:
left = spark.createDataFrame([(1,), (1,), (2,), (3,)], ["value"])
right = spark.createDataFrame([(1,), (2,), (4,)], ["value"])
print("union all:", left.union(right).count())
print("intersect:", left.intersect(right).orderBy("value").collect())
print("except all:", left.exceptAll(right).orderBy("value").collect())


## 23. `unionByName` and schema alignment

By-name union prevents accidental positional swaps. `allowMissingColumns`
can fill absent columns in upstream Spark; explicit alignment remains the
clearest contract and works consistently across Connect implementations.


In [ ]:
first_feed = spark.createDataFrame([(1, "store")], ["id", "channel"])
second_feed = spark.createDataFrame([("web", 2)], ["channel", "id"])
combined = first_feed.unionByName(second_feed)
combined.orderBy("id").show()


## 24. Partitioning and hints

`repartition` shuffles to increase or redistribute partitions; `coalesce`
usually reduces partitions without a full shuffle. Hints express optimizer
intent, but statistics and automatic planning should choose strategies in
production.


In [ ]:
partitioned = orders.repartition(4, "customer_id")
reduced = partitioned.coalesce(2)
assert partitioned.rdd.getNumPartitions() == 4
assert reduced.rdd.getNumPartitions() == 2
print({
    "repartitioned": partitioned.rdd.getNumPartitions(),
    "coalesced": reduced.rdd.getNumPartitions(),
})
reduced.hint("broadcast").limit(2).show()


## 25. Temporary views and catalog-oriented access

DataFrames can register session or global temporary views. `spark.table`
resolves a relation by name. Full catalog mutation and metastore behavior
depend on server configuration and authorization.


In [ ]:
customers.createOrReplaceTempView("api_customers")
spark.table("api_customers").select("customer_id", "name").orderBy(
    "customer_id"
).show()


## 26. Plans and diagnostics

`explain` reports parsed, analyzed, optimized, and physical plans depending
on mode. Plans establish mechanism; benchmark measurements establish cost.
The Connect server may expose a subset of classic modes.


In [ ]:
plan_target = orders.filter(F.col("status") == "paid").groupBy(
    "customer_id"
).agg(F.sum("amount").alias("total"))
plan_target.explain(mode="extended")


## 27. Actions and bounded driver collection

Actions include `show`, `count`, `collect`, `first`, `head`, `take`,
`toLocalIterator`, writes, and streaming starts. Limit or aggregate before
moving data to the driver.


In [ ]:
bounded_rows = orders.orderBy("order_id").limit(3).collect()
assert len(bounded_rows) == 3
for row in bounded_rows:
    print(row.asDict(recursive=True))


## 28. `Row`, SQL types, and schema utilities

`Row` is tuple-like and name-addressable. `StructType`/`StructField` compose
nested contracts. Types support JSON descriptions used by storage formats,
catalogs, and schema registries.


In [ ]:
example_row = Row(customer_id=10, name="Ada")
nested_type = T.StructType([
    T.StructField("id", T.LongType(), False),
    T.StructField("labels", T.ArrayType(T.StringType()), True),
])
assert example_row["name"] == "Ada"
print(nested_type.simpleString())
print(nested_type.json())


## 29. Readers and writers

`spark.read` configures batch sources such as parquet, JSON, CSV, ORC, JDBC,
text, and tables. `DataFrame.write` and `writeTo` configure output. This
Fabric notebook reference avoids filesystem mutation; the attached Lakehouse supports governed write examples.


In [ ]:
fabric_io_path = "Files/sparkrust_runtime2_roundtrip.parquet"
try:
    source_df = orders.select("order_id", "amount")
    source_df.write.mode("overwrite").parquet(fabric_io_path)
    io_rows = spark.read.parquet(fabric_io_path).orderBy("order_id").collect()
    assert len(io_rows) == source_df.count()
    print({"fabric_path": fabric_io_path, "rows": len(io_rows)})
finally:
    from notebookutils import mssparkutils
    mssparkutils.fs.rm(fabric_io_path, True)


## 30. Structured Streaming API map

Streaming uses `readStream`, `writeStream`, `StreamingQuery`, listeners,
triggers, watermarks, stateful operators, and `foreachBatch`. A live stream
needs a long-running source and sink, so use **18_structured_streaming** for
the complete runnable reference. This cell verifies the API surface without
starting an unbounded query.


In [ ]:
print({
    "readStream": hasattr(spark, "readStream"),
    "streams": hasattr(spark, "streams"),
    "writeStream": hasattr(type(orders), "writeStream"),
    "isStreaming": orders.isStreaming,
})


## 31. pandas API on Spark and Arrow interop

`pyspark.pandas` mirrors much of pandas while distributing execution. It
requires pandas/Arrow compatibility and may use APIs unavailable through a
minimal Connect endpoint. Prefer DataFrame APIs for portable remote plans;
use pandas API on Spark when pandas semantics are the application contract.


In [ ]:

print({
    "pyspark.pandas_available": importlib.util.find_spec("pyspark.pandas")
    is not None,
    "connect_mode": False,
})


## 32. MLlib, RDDs, resources, and profiling

The official index also covers `pyspark.ml`, legacy `pyspark.mllib`, RDDs,
resource profiles, status tracking, accumulators, profiler APIs, and task
contexts. RDD/SparkContext APIs are classic-driver only. Use the runnable
**17_machine_learning** and **16_rdd_dataset_compatibility** notebooks for
those families.


In [ ]:

families = ["pyspark.ml", "pyspark.mllib", "pyspark.resource"]
print({name: importlib.util.find_spec(name) is not None for name in families})
print("SparkContext available in remote mode:", not False)


## 33. Errors and testing utilities

PySpark exposes structured exception classes and testing helpers such as
`assertDataFrameEqual` and `assertSchemaEqual`. In this image, importing
`pyspark.testing` reaches an incompatible pandas-on-Spark dependency, so the
executable fallback compares schemas and bounded rows directly. Compatible
pandas/Arrow installations should use the official helpers for richer diffs.


In [ ]:
from pyspark.errors import AnalysisException, PySparkException

expected = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "value"])
actual = expected.orderBy("id")
assert actual.schema == expected.schema
assert actual.collect() == expected.collect()
print({
    "structured_errors": [AnalysisException.__name__, PySparkException.__name__],
    "testing_module_available": importlib.util.find_spec("pyspark.testing")
    is not None,
})


## 34. Spark 4.1 relational APIs: TVFs, unpivot, grouping sets, and lateral logic

`spark.tvf` exposes table-valued functions such as `range`, `explode`,
`inline`, `json_tuple`, `posexplode`, and `stack`. `unpivot`/`melt` reshape
wide data. `groupingSets`, `lateralJoin`, `exists`, and `asTable` cover
advanced relational composition; support depends on the client/server pair.


In [ ]:
tvf_rows = [row.id for row in spark.tvf.range(1, 4).collect()]
exploded = [
    tuple(row)
    for row in spark.tvf.explode(F.array(F.lit(1), F.lit(2))).collect()
]
position_exploded = [
    tuple(row)
    for row in spark.tvf.posexplode(
        F.array(F.lit("x"), F.lit("y"))
    ).collect()
]
wide = spark.createDataFrame([(1, 10, 20)], ["id", "q1", "q2"])
unpivot_rows = [
    tuple(row)
    for row in wide.unpivot(
        "id", ["q1", "q2"], "quarter", "value"
    ).orderBy("quarter").collect()
]
assert tvf_rows == [1, 2, 3]
assert exploded == [(1,), (2,)]
assert position_exploded == [(0, "x"), (1, "y")]
assert unpivot_rows == [(1, "q1", 10), (1, "q2", 20)]
print({
    "tvf_range": tvf_rows,
    "tvf_explode": exploded,
    "tvf_posexplode": position_exploded,
    "unpivot": unpivot_rows,
    "compatibility": {
        "groupingSets": "supported by Fabric Spark",
        "lateralJoin": "supported by Fabric Spark",
        "exists": "supported by Fabric Spark",
        "tvf_stack": "available when supported by the Fabric runtime",
    },
})


## 35. Observations, execution metadata, semantic plans, and operation control

`Observation`/`observe` attach aggregate metrics without changing rows.
`executionInfo`, `semanticHash`, and `sameSemantics` support diagnostics and
plan comparison. Session tags, progress handlers, artifacts, and interrupt
methods control remote operations; they are lifecycle APIs, not expressions.


In [ ]:
observed = orders.observe("api_metrics", F.count("*").alias("rows"))
observed_rows = observed.orderBy("order_id").collect()
semantic_hash = orders.semanticHash()
spark.addTag("api-reference")
tags = spark.getTags()
spark.clearTags()
assert len(observed_rows) == 5
assert isinstance(semantic_hash, int)
assert orders.sameSemantics(orders)
assert "api-reference" in tags
print({
    "semantic_hash": semantic_hash,
    "projecting_same_columns_is_same_plan": orders.sameSemantics(
        orders.select(*orders.columns)
    ),
    "execution_info": hasattr(type(observed), "executionInfo"),
    "observation_metrics_available": False,
    "tags": sorted(tags),
    "progress_handler": "Spark Connect only",
})


## 36. Statistics, sampling, catalog metadata, and Arrow collection

`DataFrameStatFunctions` and DataFrame statistics cover correlation,
contingency tables, frequent items, stratified sampling, covariance, and
quantiles. Catalog APIs inspect tables/functions/databases. `toArrow` is a
bounded columnar client action; always aggregate or limit first in production.


In [ ]:
stat_data = spark.createDataFrame(
    [(1, "a", 10.0), (1, "b", 20.0), (2, "a", 30.0)],
    ["id", "category", "value"],
)
correlation = stat_data.corr("id", "value")
frequent = set(stat_data.freqItems(["category"], 0.2).first()[0])
sampled_count = stat_data.stat.sampleBy(
    "category", {"a": 1.0, "b": 0.0}, 7
).count()
cross_tab = stat_data.crosstab("id", "category").count()
arrow_table = stat_data.orderBy("value").toArrow()
table_metadata = spark.catalog.getTable("api_orders")
assert round(correlation, 6) == 0.866025
assert frequent == {"a", "b"}
assert sampled_count == 2 and cross_tab == 2
assert arrow_table.num_rows == 3 and arrow_table.num_columns == 3
assert table_metadata.name == "api_orders"
print({
    "corr": correlation,
    "frequent_items": sorted(frequent),
    "sampled_rows": sampled_count,
    "crosstab_rows": cross_tab,
    "arrow_shape": (arrow_table.num_rows, arrow_table.num_columns),
    "catalog_table": table_metadata.name,
    "known_gaps": {
        "approxQuantile": "4.0.1 client / 4.1 Fabric runtime response-shape mismatch",
        "Catalog.listTables": "internal IndexError on this endpoint",
    },
})


## 37. Writer V2, merge, change feeds, and catalog-dependent mutations

`writeTo` returns `DataFrameWriterV2` with create/replace/append/overwrite
operations. `mergeInto` builds matched/not-matched actions and optional schema
evolution. These mutate named catalog tables, so this shared reference checks
contracts without altering a catalog. `DataFrameReader.changes` is a newer
change-feed API and may be absent from an older Connect client.


In [ ]:
writer_v2 = orders.writeTo("catalog.schema.table")
assert hasattr(writer_v2, "create") and hasattr(writer_v2, "append")
assert hasattr(orders, "mergeInto")
print({
    "writer_v2": {
        name: hasattr(writer_v2, name)
        for name in (
            "using", "option", "tableProperty", "partitionedBy", "create",
            "replace", "createOrReplace", "append", "overwrite",
            "overwritePartitions",
        )
    },
    "merge_into": hasattr(orders, "mergeInto"),
    "reader_changes": hasattr(spark.read, "changes"),
    "mutation_executed": False,
})


## 38. Python data sources, UDF/UDTF registration, and Arrow batch APIs

PySpark 4.1 includes Python `DataSource` reader/writer/streaming interfaces,
scalar/Pandas/Arrow UDFs, UDTFs, `mapInPandas`, `applyInPandas`, and
`mapInArrow`. They require matching Python dependencies and server features.
The Spark User Guide notebook performs executable UDF/UDTF diagnostics; this
reference maps the complete family without starting workers or streams.


In [ ]:
python_execution_apis = {
    "data_source_registration": hasattr(spark, "dataSource"),
    "udf_registration": hasattr(spark, "udf"),
    "udtf_registration": hasattr(spark, "udtf"),
    "mapInPandas": hasattr(orders, "mapInPandas"),
    "mapInArrow": hasattr(orders, "mapInArrow"),
    "transformWithStateInPandas": hasattr(
        orders, "transformWithStateInPandas"
    ),
    "pyspark.sql.datasource": importlib.util.find_spec(
        "pyspark.sql.datasource"
    ) is not None,
}
assert python_execution_apis["data_source_registration"]
assert python_execution_apis["mapInPandas"]
print(python_execution_apis)


## 39. Variant values, table arguments, pipelines, logging, and version skew

The official 4.1 index also contains `VariantVal`, `TableArg`, declarative
`pyspark.pipelines`, and `PySparkLogger`. These surfaces are client-version or
deployment dependent. This Fabric notebook intentionally runs a PySpark 4.0.1
client against a Spark 4.1 server, so capability checks are part of the API
contract: server support cannot create a missing client method.


In [ ]:
optional_modern_families = {
    name: importlib.util.find_spec(name) is not None
    for name in (
        "pyspark.sql.datasource", "pyspark.sql.table_arg",
        "pyspark.pipelines", "pyspark.logger", "pyspark.testing",
    )
}
print({
    "fabric_runtime": "2.0",
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "VariantVal": hasattr(__import__("pyspark.sql", fromlist=["VariantVal"]), "VariantVal"),
    "families": optional_modern_families,
})


## 40. Capstone: compose, validate, and clean up

This pipeline joins dimensions, derives nested values, computes grouped
metrics and a rank, validates invariants, and performs one bounded action.
The final `stop` demonstrates explicit session ownership.


In [ ]:
enriched = orders.join(customers, "customer_id", "left").select(
    "order_id", "customer_id", "name", "region", "amount", "attrs"
).withColumn(
    "customer", F.coalesce("name", F.lit("Guest"))
).withColumn(
    "device", F.expr("element_at(attrs, 'device')")
)
summary = enriched.groupBy("region", "device").agg(
    F.count(F.lit(1)).alias("orders"),
    F.sum("amount").alias("net_amount"),
).withColumn(
    "regional_rank",
    F.dense_rank().over(
        Window.partitionBy("region").orderBy(F.col("net_amount").desc())
    ),
).orderBy(F.col("region").asc_nulls_last(), "regional_rank", "device")
rows = summary.limit(20).collect()
assert rows and sum(row.orders for row in rows) == 5
for row in rows:
    print(row.asDict())
# Fabric owns the Spark session; do not call spark.stop().


## Reference checklist

- Build expressions lazily; know which calls are actions.
- Define schemas and decimal widths at system boundaries.
- Treat nullability, time zones, and ordering as semantics.
- Validate join multiplicity before aggregating.
- Use deterministic survivor and top-K rules.
- Bound every driver action.
- Inspect plans before tuning and benchmark under matched resources.
- Prefer automatic statistics-driven planning over manual hints.
- Use dedicated streaming, ML, RDD, and lakehouse notebooks for APIs whose
  lifecycle cannot be represented by a short bounded reference cell.
